# 💼 Employee Salary Prediction Using Machine Learning
### Gexton Education — Data Science Internship Program | Task #06
**Supervised by:** Sir Muhammad Arham MH

**Dataset:** Employee Salary Dataset — Kaggle  
**Source:** https://www.kaggle.com/datasets/prince7489/employee-salary-dataset/data

---
**Objective:** Predict employee monthly salaries using ML based on experience, education, department, city, age, and gender.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib, warnings; warnings.filterwarnings('ignore')
%matplotlib inline
BASE = Path('..').resolve()
print('Setup complete ✓')

## 1. Data Loading & Understanding
> **Source:** [Kaggle — Employee Salary Dataset](https://www.kaggle.com/datasets/prince7489/employee-salary-dataset/data)


In [ ]:
df_raw = pd.read_csv(BASE / 'data' / 'employee_salary_dataset.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(10)

In [ ]:
print('Data Types:'); print(df_raw.dtypes)
print('\nMissing:', df_raw.isnull().sum().sum())
print('Duplicates:', df_raw.duplicated().sum())
df_raw.describe()

In [ ]:
for col in ['Department','Education_Level','Gender','City']:
    print(f'\n{col}:'); print(df_raw[col].value_counts())

## 2. Data Cleaning


In [ ]:
df = df_raw.copy()
df = df.drop(columns=['Name'])  # Drop PII column
edu_order = {'High School':0,'Bachelor':1,'Master':2,'PhD':3}
df['education_encoded']    = df['Education_Level'].map(edu_order)
df['gender_encoded']       = (df['Gender']=='Male').astype(int)
dept_map = {d:i for i,d in enumerate(sorted(df['Department'].unique()))}
city_map = {c:i for i,c in enumerate(sorted(df['City'].unique()))}
df['department_encoded']   = df['Department'].map(dept_map)
df['city_encoded']         = df['City'].map(city_map)
df.to_csv(BASE / 'data' / 'cleaned_employee_salary.csv', index=False)
print(f'Cleaned shape: {df.shape}'); df.head()

## 3. Exploratory Data Analysis (EDA)


### 3.1 Experience vs Monthly Salary


In [ ]:
edu_pal = {'High School':'#E74C3C','Bachelor':'#3498DB','Master':'#27AE60','PhD':'#9B59B6'}
fig, ax = plt.subplots(figsize=(11,5))
for edu, clr in edu_pal.items():
    m = df['Education_Level']==edu
    ax.scatter(df.loc[m,'Experience_Years'],df.loc[m,'Monthly_Salary'],color=clr,s=70,alpha=0.8,label=edu)
z=np.polyfit(df['Experience_Years'],df['Monthly_Salary'],1)
xs=np.linspace(0,20,100)
ax.plot(xs,np.poly1d(z)(xs),'k--',lw=2,label='Trend')
ax.set_title('Experience vs Monthly Salary'); ax.set_xlabel('Experience (yrs)')
ax.set_ylabel('Monthly Salary (₹)'); ax.legend(); plt.tight_layout(); plt.show()
print(f'Correlation: r = {df["Experience_Years"].corr(df["Monthly_Salary"]):.3f}')

### 3.2 Salary by Department


In [ ]:
dept_order = df.groupby('Department')['Monthly_Salary'].median().sort_values().index
df.boxplot(column='Monthly_Salary',by='Department',figsize=(11,5),
           positions=range(len(dept_order)),vert=False)
plt.title('Monthly Salary by Department'); plt.suptitle('')
plt.tight_layout(); plt.show()

### 3.3 Education Level Impact


In [ ]:
edu_stats = df.groupby('Education_Level')['Monthly_Salary'].agg(['mean','median'])\
             .reindex(['High School','Bachelor','Master','PhD'])
print(edu_stats.round(0).to_string())
edu_stats.plot(kind='bar',figsize=(9,5),color=['#1E3A5F','#E07B39'],edgecolor='white',alpha=0.88)
plt.title('Salary by Education Level'); plt.xticks(rotation=0)
plt.legend(['Mean','Median']); plt.tight_layout(); plt.show()

### 3.4 Age vs Monthly Salary


In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
sc=ax.scatter(df['Age'],df['Monthly_Salary'],c=df['Experience_Years'],
             cmap='YlOrRd',s=80,alpha=0.8,edgecolors='white')
plt.colorbar(sc,ax=ax,label='Experience (yrs)')
ax.set_title('Age vs Monthly Salary'); ax.set_xlabel('Age'); ax.set_ylabel('Monthly Salary (₹)')
plt.tight_layout(); plt.show()
print(f'Age-Salary correlation: r = {df["Age"].corr(df["Monthly_Salary"]):.3f}')

### 3.5 Highest & Lowest Paying Positions (Department & City)


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
dept_avg = df.groupby('Department')['Monthly_Salary'].mean().sort_values()
city_avg = df.groupby('City')['Monthly_Salary'].mean().sort_values()
dept_avg.plot(kind='barh',ax=axes[0],color='#1E3A5F',alpha=0.85,edgecolor='white')
city_avg.plot(kind='barh',ax=axes[1],color='#E07B39',alpha=0.85,edgecolor='white')
axes[0].set_title('Avg Salary by Department')
axes[1].set_title('Avg Salary by City')
plt.tight_layout(); plt.show()
print(f'Top dept: {dept_avg.idxmax()} ₹{dept_avg.max():,.0f}')
print(f'Top city: {city_avg.idxmax()} ₹{city_avg.max():,.0f}')

## 4. Machine Learning Models


In [ ]:
NUMERIC = ['Age','Experience_Years']
CATEG   = ['Department','Education_Level','Gender','City']
TARGET  = 'Monthly_Salary'
X = df[NUMERIC+CATEG]; y = df[TARGET]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
pre = ColumnTransformer([('num',StandardScaler(),NUMERIC),
                         ('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),CATEG)])
models = {
    'Linear Regression': Pipeline([('pre',pre),('m',LinearRegression())]),
    'Decision Tree':     Pipeline([('pre',pre),('m',DecisionTreeRegressor(max_depth=4,random_state=42))]),
    'Random Forest':     Pipeline([('pre',pre),('m',RandomForestRegressor(n_estimators=100,max_depth=4,random_state=42))]),
}
for name,pipe in models.items():
    pipe.fit(X_train,y_train); print(f'{name}: trained ✓')

## 5. Model Evaluation


In [ ]:
rows=[]
for name,pipe in models.items():
    yp=pipe.predict(X_test)
    cv=cross_val_score(pipe,X,y,cv=5,scoring='r2')
    rows.append({'Model':name,'MAE(₹)':round(mean_absolute_error(y_test,yp),0),
                 'RMSE(₹)':round(np.sqrt(mean_squared_error(y_test,yp)),0),
                 'R² Test':round(r2_score(y_test,yp),4),
                 'CV R² Mean':round(cv.mean(),4)})
res=pd.DataFrame(rows)
print(res.to_string(index=False))
print('\n⚠ Low R² due to small dataset (50 records). Larger data will improve accuracy.')

## 6. Key Findings & Conclusion
- **Experience** is the strongest salary driver (RF importance: 47%)
- **Marketing** is the highest-paying department (₹96,431/mo avg)
- **Finance** is the lowest-paying department (₹67,262/mo avg)
- **Bangalore** is the highest-paying city (₹99,004/mo avg)
- **Low R² scores** reflect the small dataset size (50 rows) — not a model flaw
- With 500+ records, accuracy would improve significantly
- **Linear Regression** performs best (highest test R²), consistent with weak linear trends

**Dataset Source:** [Kaggle — Employee Salary Dataset by prince7489](https://www.kaggle.com/datasets/prince7489/employee-salary-dataset/data)
